<p align="left">
  <img src="https://www.gap-labs.net/gap-labs.png" width="140">
</p>

# Team Structures Demo
Interactive demo that runs directly in Google Colab.

**GAP Labs makes incentive risk measurable before it appears as project failure.**

A compact simulation of incentive structures using  
`capitalmarket.capitalselector`.

### Quick impact snapshot

- A compact snapshot table is computed automatically in **Detail Analysis**.
- It tracks KPI-value gap, true-value delta vs healthy baseline, and gaming ratio.
- Numbers adapt to the active run profile (`colab-fast` or `balanced`).

### Why this matters for teams and leadership

- **Early warning:** detect KPI-value divergence before operational damage accumulates.
- **Scenario stress-testing:** compare governance regimes before changing incentives in production.
- **Governance tuning:** identify which policy levers reduce gaming and protect true project value.

© 2026 GAP Labs GmbH. All rights reserved.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gap-labs/meta-credit-dynamics/blob/main/notebooks/team_demo.ipynb)

### Running the Demo

1. Open the notebook in **Google Colab** using the badge above.
2. Select **Runtime -> Run all**.

*Note: Colab does not allow automatic execution via URL parameters.*

### License

Project license:  
https://github.com/gap-labs/meta-credit-dynamics?tab=License-1-ov-file

## Setup

The following cells initialise the simulation environment
and load the model used in the demo.

In [ ]:
# Basic environment setup

import os, sys, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
PUBLIC_REPO_URL = "https://github.com/gap-labs/meta-credit-dynamics.git"
COLAB_REPO_DIR = Path("/content/meta-credit-dynamics")

if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", PUBLIC_REPO_URL, str(COLAB_REPO_DIR)], check=True)
    req_file = COLAB_REPO_DIR / "requirements.txt"
    if req_file.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)], check=True)
    PROJECT_IMPORT_ROOT = str(COLAB_REPO_DIR)
else:
    # set this to your extracted repo root (where "capitalmarket/" lives)
    PROJECT_IMPORT_ROOT = "~/prj/dl"

if PROJECT_IMPORT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_IMPORT_ROOT)

from capitalmarket.capitalselector.builder import CapitalSelectorBuilder
from capitalmarket.capitalselector.population_manager import PopulationManager, RebirthConfig
from capitalmarket.capitalselector.determinism import enable_determinism
from capitalmarket.capitalselector.worlds import GovernanceWorld

## Simulation Model

The simulation models a project environment in which teams respond to incentives.

Two key signals are tracked over time:

**True Value V(t)**  
The real value created by the project.

**Rewarded KPI K(t)**  
The metric used to evaluate performance.

Teams respond rationally to incentives and optimise what is rewarded.

If incentives remain aligned, both signals evolve together.

If incentives drift, teams may continue to improve the KPI
even while the underlying project value deteriorates.

In [ ]:
REGIMES = {
    "Healthy Incentives": dict(alpha=0.82, manipulability=0.08, reweight_speed=0.02, punishment=0.08, volatility=0.05),
    "Metric Gaming": dict(alpha=0.45, manipulability=0.75, reweight_speed=0.60, punishment=0.20, volatility=0.12),
    "Punitive Governance": dict(alpha=0.60, manipulability=0.45, reweight_speed=0.35, punishment=0.90, volatility=0.15),
    "Unstable Metrics": dict(alpha=0.58, manipulability=0.50, reweight_speed=0.45, punishment=0.30, volatility=0.90),
}

def corr(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if len(a) < 2 or np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

def alignment_index(V: np.ndarray, K: np.ndarray) -> float:
    return corr(V, K)

def gaming_ratio(dK: np.ndarray, dV: np.ndarray) -> float:
    dK = np.asarray(dK)
    dV = np.asarray(dV)
    pos = dK > 0
    if pos.sum() == 0:
        return 0.0
    return float(((dK > 0) & (dV <= 0)).sum() / pos.sum())

def gini(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    if np.allclose(x.sum(), 0):
        return 0.0
    x = np.sort(np.maximum(x, 0))
    n = x.size
    idx = np.arange(1, n + 1)
    return float((2 * np.sum(idx * x) / (n * np.sum(x))) - (n + 1) / n)

def governance_volatility(omega_hist: np.ndarray) -> float:
    omega_hist = np.asarray(omega_hist, dtype=float)
    if len(omega_hist) < 2:
        return 0.0
    diffs = omega_hist[1:] - omega_hist[:-1]
    return float(np.mean(np.linalg.norm(diffs, axis=1)))

def structural_churn(rebirth_counts: np.ndarray, T: int) -> float:
    return float(np.sum(rebirth_counts) / max(T, 1))

def index_report(V, K, dK, dV, wealth, omega_hist, rebirth_counts):
    return {
        "Alignment Index (AI)": alignment_index(V, K),
        "Gaming Ratio (GR)": gaming_ratio(dK, dV),
        "Concentration Index (CI, Gini)": gini(wealth),
        "Governance Volatility (GV)": governance_volatility(omega_hist),
        "Structural Churn (SC)": structural_churn(rebirth_counts, len(V)),
    }

def simulate_project(regime: dict, *, N=120, T=2000, K_channels=8, seed=42):
    enable_determinism(seed)

    # initial population: N processes with selectors
    processes = {}
    for pid in range(N):
        sel = CapitalSelectorBuilder().with_K(K_channels).build()
        processes[int(pid)] = sel

    manager = PopulationManager(
        processes=processes,
        rebirth_config=RebirthConfig(
            enabled=True,
            base_liquidity=0.20,  # keeps rebirth "alive"
            eta=0.10,
            kappa=1.0,
            max_claims_per_process=1_000_000,
        ),
        backend="cpu",
    )

    world = GovernanceWorld(K_channels=K_channels, regime=regime, seed=seed)

    # histories for indices
    omega_hist = []
    wealth_hist = []
    rebirth_counts = np.zeros(T, dtype=int)

    for t in range(T):
        active_pids = sorted([pid for pid, sel in manager.processes.items() if not bool(getattr(sel, "dead", False))])

        events = world.step(t, active_pids)
        # PopulationManager expects process_events mapping: pid -> {"r_vec","c_total","freeze"}
        out = manager.step_tau(tau=int(t), process_events=events, jackpot=0.0)

        rebirth_counts[t] = len(out.get("newborn_ids", []))

        # sample system state: mean wealth + mean weights over living
        living = [manager.processes[pid] for pid in active_pids]
        if living:
            wealth_hist.append(float(np.mean([float(s.wealth) for s in living])))
            omega_hist.append(np.mean([s.w for s in living if getattr(s, "w", None) is not None], axis=0))
        else:
            wealth_hist.append(float("nan"))
            omega_hist.append(np.full(K_channels, np.nan))

    omega_hist = np.asarray(omega_hist, dtype=float)
    wealth_final = np.array([float(s.wealth) for s in manager.processes.values()], dtype=float)

    V = np.asarray(world.V_hist, dtype=float)
    M = np.asarray(world.M_hist, dtype=float)
    K = np.asarray(world.K_hist, dtype=float)

    dV = np.diff(np.r_[0.0, V])
    dK = np.diff(np.r_[0.0, K])

    return {
        "V": V, "M": M, "K": K,
        "dV": dV, "dK": dK,
        "wealth": wealth_final,
        "omega_hist": omega_hist,
        "rebirth_counts": rebirth_counts,
        "wealth_mean_t": np.asarray(wealth_hist, dtype=float),
    }

RUN_PRESETS = {
    "colab-fast": {
        "single": dict(N=80, T=800, K_channels=8, seed=42),
        "compare": dict(N=80, T=1200, K_channels=8, seed=42),
    },
    "balanced": {
        "single": dict(N=120, T=1200, K_channels=8, seed=42),
        "compare": dict(N=120, T=2000, K_channels=8, seed=42),
    },
}

USER_RUN_PROFILE = None  # set to "colab-fast" or "balanced" to override auto selection

if USER_RUN_PROFILE is None:
    RUN_PROFILE = "colab-fast" if IN_COLAB else "balanced"
else:
    if USER_RUN_PROFILE not in RUN_PRESETS:
        raise ValueError(f"Unknown run profile: {USER_RUN_PROFILE}. Choose one of: {list(RUN_PRESETS)}")
    RUN_PROFILE = USER_RUN_PROFILE
SINGLE_RUN_CFG = dict(RUN_PRESETS[RUN_PROFILE]["single"])
COMPARE_RUN_CFG = dict(RUN_PRESETS[RUN_PROFILE]["compare"])
print(f"Run profile: {RUN_PROFILE} | single={SINGLE_RUN_CFG} | compare={COMPARE_RUN_CFG}")
print("Set USER_RUN_PROFILE = 'colab-fast' or 'balanced' to override auto profile")

## Example: Metric Gaming Regime

This example shows how the system behaves when teams
gradually learn to optimise the rewarded metric.

In [ ]:
REGIME_NAME = "Metric Gaming"
regime = REGIMES[REGIME_NAME]

sim = simulate_project(regime, **SINGLE_RUN_CFG)

report = index_report(sim["V"], sim["K"], sim["dK"], sim["dV"], sim["wealth"], sim["omega_hist"], sim["rebirth_counts"])
pd.Series(report).to_frame("value")

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(sim["V"], label="True Value V(t)")
plt.plot(sim["K"], label="Rewarded KPI K(t)")
plt.legend()
plt.title(f"KPI vs True Project Value {REGIME_NAME}")
plt.show()

plt.figure(figsize=(8,4))
plt.plot(sim["wealth_mean_t"], label="Mean wealth (living)")
plt.legend()
plt.title(f"{REGIME_NAME}: Mean wealth over time")
plt.show()

plt.figure(figsize=(8,4))
plt.plot(np.cumsum(sim["rebirth_counts"]), label="Cumulative rebirths")
plt.legend()
plt.title(f"{REGIME_NAME}: Structural churn")
plt.show()

## Hero Chart: KPI-Value Misalignment

`Dashboard Signal vs Real Project Value` plots `KPI - True Value` over time.

- Positive values: the dashboard overestimates real project value.
- Negative values: the dashboard underestimates real project value.
- A widening distance from zero indicates growing misalignment risk.

In [ ]:
plt.figure(figsize=(8,4))

gap = sim["K"] - sim["V"]

plt.plot(gap, color="darkred")
plt.axhline(0, linestyle="--", color="black", alpha=0.5)

plt.title("Dashboard Signal vs Real Project Value")
plt.ylabel("KPI – True Value")
plt.xlabel("Time")

note = (
    "Positive gap: KPI overestimates real value"
    if float(np.nanmean(gap)) >= 0.0
    else "Negative gap: KPI underestimates real value"
)
plt.text(
    0.5,
    0.93,
    note,
    transform=plt.gca().transAxes,
    fontsize=9,
    ha="center",
    va="top",
)

plt.show()

## Detail Analysis

In [ ]:
RUN_CFG = dict(COMPARE_RUN_CFG)

regime_signature = tuple(
    (name, tuple(sorted(cfg.items())))
    for name, cfg in sorted(REGIMES.items())
)
cache_key = (tuple(sorted(RUN_CFG.items())), regime_signature)

if globals().get("_sim_cache_key") == cache_key:
    all_sims = globals()["_sim_cache_sims"]
    all_reports = globals()["_sim_cache_reports"]
    print("Using cached simulation results")
else:
    all_sims = {}
    all_reports = {}

    for regime_name, regime_cfg in REGIMES.items():
        sim_i = simulate_project(regime_cfg, **RUN_CFG)
        report_i = index_report(
            sim_i["V"], sim_i["K"], sim_i["dK"], sim_i["dV"],
            sim_i["wealth"], sim_i["omega_hist"], sim_i["rebirth_counts"]
        )
        all_sims[regime_name] = sim_i
        all_reports[regime_name] = report_i

    globals()["_sim_cache_key"] = cache_key
    globals()["_sim_cache_sims"] = all_sims
    globals()["_sim_cache_reports"] = all_reports

metric_order = [
    "Alignment Index (AI)",
    "Gaming Ratio (GR)",
    "Concentration Index (CI, Gini)",
    "Governance Volatility (GV)",
    "Structural Churn (SC)",
]

metrics_df = pd.DataFrame(all_reports).T[metric_order]

healthy_v_final = float(all_sims["Healthy Incentives"]["V"][-1])
metric_v_final = float(all_sims["Metric Gaming"]["V"][-1])
metric_gap_final = float(all_sims["Metric Gaming"]["K"][-1] - all_sims["Metric Gaming"]["V"][-1])
metric_v_delta_pct = 100.0 * (metric_v_final / healthy_v_final - 1.0)

snapshot_df = pd.DataFrame(
    {
        "Metric": [
            "Metric Gaming final KPI-value gap (K - V)",
            "Metric Gaming final true value vs Healthy Incentives",
            "Metric Gaming gaming ratio",
        ],
        "Value": [
            f"{metric_gap_final:.1f}",
            f"{metric_v_delta_pct:+.1f}%",
            f"{float(metrics_df.loc['Metric Gaming', 'Gaming Ratio (GR)']):.3f}",
        ],
    }
)

snapshot_df

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Comparison 1: AI vs GR (bubble size = Structural Churn)
ax = axes[0]
for regime_name, row in metrics_df.iterrows():
    churn = float(row["Structural Churn (SC)"])
    ax.scatter(
        float(row["Alignment Index (AI)"]),
        float(row["Gaming Ratio (GR)"]),
        s=400 + 12000 * churn,
        alpha=0.7,
    )
    ax.annotate(regime_name, (float(row["Alignment Index (AI)"]), float(row["Gaming Ratio (GR)"])), xytext=(6, 4), textcoords="offset points")

ax.set_title("Regime comparison: Alignment vs Gaming")
ax.set_xlabel("Alignment Index (higher is better)")
ax.set_ylabel("Gaming Ratio (lower is better)")
ax.grid(alpha=0.25)

# Comparison 2: KPI-Value gap over time (K - V)
ax = axes[1]
for regime_name, sim_i in all_sims.items():
    gap = np.asarray(sim_i["K"], dtype=float) - np.asarray(sim_i["V"], dtype=float)
    ax.plot(gap, label=regime_name, linewidth=1.4)

ax.axhline(0.0, color="black", linewidth=1.0, alpha=0.6)
ax.set_title("KPI-Value gap over time")
ax.set_xlabel("t")
ax.set_ylabel("K(t) - V(t)")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)

fig.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

for regime_name, sim_i in all_sims.items():
    plt.plot(sim_i["V"], label=regime_name)

plt.title("True Project Value across Incentive Regimes")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(6,6))

for regime_name, sim_i in all_sims.items():
    plt.scatter(sim_i["K"], sim_i["V"], alpha=0.3, label=regime_name)

plt.xlabel("Rewarded KPI K(t)")
plt.ylabel("True Project Value V(t)")
plt.legend()
plt.title("Relationship between KPI and True Project Value")
plt.show()

## Executive Summary

### What this shows

Teams optimize what is rewarded.

When incentives remain aligned, rewarded KPIs track true project value.
When incentives drift, KPI performance can improve while real project value deteriorates.

This means healthy dashboards can coexist with declining real outcomes.

### Interpretation boundary

This is an illustrative scenario model, not a historical backtest or a full organizational digital twin.

Several real-world mechanisms are intentionally simplified, including deeper organizational feedback loops, changing leadership incentives, and richer behavioral dynamics.

### Business next step

- Use this view as an early warning signal for incentive-risk drift.
- Compare governance options before operational rollout.
- Turn KPI drift into an explicit leadership decision trigger.

If you want to apply this to a real project portfolio, contact us for a 60-minute incentive-risk review:

**contact@gap-labs.net**

© 2026 GAP Labs GmbH